In [10]:
from datasets import load_dataset
from components.models.sentiment_analysis.Dlsta import Dlsta
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np


In [11]:
def filter_single_label(example):
    if len(example["labels"]) > 0:
        example["labels"] = example["labels"][0]
    else:
        example["labels"] = -1  
    return example


In [12]:
def tokenizer(texts):
    tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)   
    return tokenizer.texts_to_sequences(texts)


In [13]:
def separate_data(data):
    texts = [item["text"] for item in data]
    labels = [item["labels"] for item in data]
    return labels, texts

In [14]:
def encode_label(labels, num_classes=28):
    y = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        y[i, label] = 1
    return y

In [15]:
def train():
    dlsta = Dlsta()
    ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    data = ds["train"]
    
    data = data.map(filter_single_label)  
    
    labels, texts = separate_data(data)
    
    y = encode_label(labels=labels)
    tokens = tokenizer(texts)
    
    x_test,y_test = dlsta.train_model( sequences=tokens, y=y)
    return dlsta

In [16]:
ia = train()

1
Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.0000e+00 - loss: 0.6916
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.0000e+00 - loss: 0.6919
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.0000e+00 - loss: 0.6890
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.0000e+00 - loss: 0.6819
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.0000e+00 - loss: 0.6757


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 142, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 142, 128)       │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 28)             │         7,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,703,446 (90.42 MB)

 Trainable params: 7,901,148 (30.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 15,802,298 (60.28 MB)

In [17]:
def test(dlsta):
    dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
    labels_names = dataset["train"].features["labels"].feature.names
    test_data = dataset["test"]
    labels, texts = separate_data(test_data)

    tokens= tokenizer(texts)

    predict = (dlsta.predict(x_text=tokens))
    for i, prediction in enumerate(predict):
        # Identifier les indices des labels prédits avec une probabilité > 0.5
        predicted_indices = [j for j, prob in enumerate(prediction) if prob > 0.5]
        
        # Si aucun label n'est au-dessus du seuil, considérer comme "neutre" ou "non identifié"
        if not predicted_indices:
            print(f"Texte {i + 1} : Aucun label identifié (Neutre ou Inconnu)")
        else:
            # Récupérer les noms des labels correspondants
            predicted_labels = [labels[idx] for idx in predicted_indices]
            
            # S'assurer que `predicted_labels` contient des chaînes avant de les afficher
            if isinstance(predicted_labels, list):
                predicted_labels = [str(label) for label in predicted_labels]
            
            print(f"Texte {i + 1} : Émotions prédites : {', '.join(predicted_labels)} avec Emotion prévues : {labels[i]}")

    
    

In [18]:
test(ia)

170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
Texte 1 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [25]
Texte 2 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [0]
Texte 3 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [13]
Texte 4 : Émotions prédites : [25], [27], [15], [14], [27] avec Emotion prévues : [15]
Texte 5 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [27]
Texte 6 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [15]
Texte 7 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [15]
Texte 8 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [15]
Texte 9 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [24]
Texte 10 : Émotions prédites : [25], [27], [15], [14], [27] avec Emotion prévues : [25]
Texte 11 : Émotions prédites : [27], [15], [14], [27] avec Emotion prévues : [3, 10]
Texte 12 : Émotions prédites : [27], [15], [14], [2

In [9]:
from components.models.sentiment_analysis.DistiBERT import DistiBERT

analyse = DistiBERT()

result = analyse.analyze(["I would like to play a game of basket.","It s time to play","I don't want to go there","I don't like this place","I hate this place"])
analyse.result(results=result,texts=["I would like to play a game of basket.","It s time to play","I don't want to go there","I don't like this place","I hate this place"])

Device set to use cpu


Texte: I would like to play a game of basket.
Sentiment: positive, Score: 0.50

Texte: It s time to play
Sentiment: positive, Score: 0.49

Texte: I don't want to go there
Sentiment: neutral, Score: 0.55

Texte: I don't like this place
Sentiment: neutral, Score: 0.52

Texte: I hate this place
Sentiment: negative, Score: 0.95

